# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vxsnth/Machine_learning/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Capstone — Refresh / Content Opportunity Scoring

## 1. Question

### Research question

Can observed search-performance signals be used to prioritize content pages for human refresh review?

### Decision supported

The goal is to produce a ranked list of content pages that an editor can review first. The output is decision-support rather than an automatic content-change system.

Each row represents a content page for a client. The output is a priority score, rank, action label, and reason code.

A wrong positive call means an editor spends time reviewing a page that does not need a refresh. A wrong negative call means a page that may deserve review is ranked too low. Because the output is used for prioritization rather than automatic action, the model is intended to support human judgment.

Data and ML are useful because multiple search and content signals can interact, making a ranked prioritization more useful than relying on a single fixed threshold.

In [ ]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{get_hf_token()}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

print("Connected successfully.")

## 2. Data

This analysis uses the FlyRank internship warehouse release.

The main source is the daily content-performance fact table. February 2026 is used as the observed feature window for the baseline and initial analysis.

The unit of analysis is a content page within a client.

The analysis uses observed search-performance fields such as Google Search Console impressions, clicks, and average position.

Label-derived fields such as `trend_direction` and `trend_pct` are deliberately excluded from model inputs because they summarize movement that may overlap with the outcome being predicted.

Pseudonymous client and content IDs are used only for grouping, joining, and reporting ranked rows. They are not model features.

Future-period outcome fields are excluded from the feature set to prevent leakage.

No client names, domains, private queries, credentials, or raw client-identifying exports are used in the public analysis.

In [ ]:
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

schema = con.sql(f"""
    DESCRIBE SELECT * FROM {FEB}
""").df()

print("February columns:")
print(schema["column_name"].tolist())

signal_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(ga4_pageviews) AS pageviews_feb,
        SUM(ga4_engaged_sessions) AS engaged_sessions_feb
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Page-level rows:", len(signal_df))
signal_df.head()

## 3. Methodology

The baseline uses February search-performance observations to produce a transparent page-priority score.

The initial signals are search volume and click-through rate (CTR). Search volume represents observed search visibility, while CTR represents the share of observed impressions that resulted in clicks.

The eventual ML model will use only information available at the decision point. Label-derived trend fields, future-period measurements, and pseudonymous IDs will not be used as predictive features.

Target definition: the model will predict a future page-level performance outcome using a later time window, with the exact label defined from the available warehouse fields and kept separate from the feature window.

The baseline provides a transparent comparison against the ML model. Both approaches will be evaluated on the same held-out data and metric.

Validation will be time-aware so that information from the future is not used to predict the past.

Leakage checks will explicitly exclude `trend_direction`, `trend_pct`, future-period measurements, and identifiers from the model feature set.

In [ ]:
signal_df["ctr_feb"] = np.where(
    signal_df["impressions_feb"] > 0,
    signal_df["clicks_feb"] / signal_df["impressions_feb"],
    np.nan
)

feature_cols = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "pageviews_feb",
    "engaged_sessions_feb",
]

print("Feature columns:")
print(feature_cols)

print("\nMissing values:")
print(signal_df[feature_cols].isna().sum())

### Baseline

The baseline is a simple transparent ranking rule using February search volume and CTR.

Pages with greater observed search volume receive greater priority because they represent more visible search opportunities. Pages with lower CTR receive greater priority because they have observed impressions but capture a smaller share of those impressions as clicks.

The baseline is intentionally simple so that the later ML model can be compared against a transparent rule rather than against an opaque or already-optimized system.

In [ ]:
baseline = signal_df.copy()

baseline["volume_score"] = baseline["impressions_feb"].rank(
    pct=True,
    ascending=True
)

baseline["ctr_rank"] = baseline["ctr_feb"].rank(
    pct=True,
    ascending=True,
    na_option="bottom"
)

baseline["ctr_score"] = 1 - baseline["ctr_rank"]

baseline["baseline_score"] = (
    0.6 * baseline["volume_score"]
    + 0.4 * baseline["ctr_score"]
)

baseline["action"] = "REFRESH_REVIEW"
baseline["reason_code"] = "HIGH_VOLUME_LOW_CTR"

baseline = baseline.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

baseline_output = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "baseline_score",
        "action",
        "reason_code",
    ]
]

print("Baseline rows:", len(baseline_output))
baseline_output.head(10)

## 4. Results (vs baseline)

The final model will be compared with the transparent baseline on the same held-out time period.

The primary evaluation will focus on ranking quality because the practical use case is prioritizing a limited number of pages for human review.

Precision@K will be reported together with the task base rate. Where appropriate, AUC or lift over the baseline will also be reported.

No final performance claim is made until the model has been trained and evaluated on the held-out period.

In [ ]:
results = pd.DataFrame({
    "method": ["Baseline"],
    "precision_at_k": [np.nan],
    "base_rate": [np.nan],
    "notes": ["Final values will be calculated after the time-aware model evaluation."]
})

results

## 5. Limitations

This analysis is decision-support, not proof of causal impact.

A high priority score does not prove that refreshing a page will improve rankings, clicks, traffic, or engagement.

Search performance can be affected by factors that are not represented in the available data, including seasonality, changes in demand, measurement availability, competition, and other contextual factors.

The ranking identifies pages that are worth human review based on observed signals. Editors should consider page intent, content quality, business context, and other information before taking action.

The model also depends on the quality and availability of the underlying search and engagement measurements.

In [ ]:
limitations_checks = {
    "Uses_future_features": False,
    "Uses_trend_direction_as_feature": False,
    "Uses_trend_pct_as_feature": False,
    "Uses_client_id_as_feature": False,
    "Uses_content_id_as_feature": False,
    "Claims_causal_refresh_impact": False,
}

limitations_table = pd.DataFrame(
    list(limitations_checks.items()),
    columns=["check", "value"]
)

limitations_table

## 6. Ranked recommendations

The recommendation output is a ranked action queue for human review.

The primary action is `REFRESH_REVIEW`. The reason code explains why a page entered the queue.

The ranking should be treated as directional prioritization. Higher-ranked pages are not guaranteed to benefit from a refresh; they are pages that the scoring system considers more worthy of review based on the observed signals.

A FlyRank editor could use the ranked list as a starting point for manual investigation rather than as an automatic publishing or pruning decision.

In [ ]:
ranked_recommendations = baseline_output.copy()

print("Top 20 recommended pages:")
ranked_recommendations.head(20)

## 7. Artifacts the paper embeds

The deployed paper will include visual evidence supporting the analysis.

Planned artifacts include:

1. A distribution of baseline scores.
2. A comparison of baseline and ML ranking performance.
3. A ranked recommendation table.
4. A feature-importance or model-interpretation chart.
5. A concise error analysis showing examples of strong and weak predictions.

All reported numbers will be generated from the notebook so that the paper remains reproducible.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.hist(
    baseline["baseline_score"].dropna(),
    bins=20
)

plt.xlabel("Baseline score")
plt.ylabel("Number of pages")
plt.title("Distribution of baseline priority scores")
plt.tight_layout()
plt.show()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.